# PharmaSim — Two-Compartment PK Simulator
### Interactive Demo Notebook

---

## The Biology

When a drug enters your body, it distributes across different tissues at different rates.
A **two-compartment model** captures this with two coupled differential equations:

```
┌─────────────────────────────────────────────────┐
│                                                 │
│   Drug input ──▶  [Central]  ⇄  [Peripheral]   │
│                    (blood)       (tissue)        │
│                       │                         │
│                       ▼ k10                     │
│                   Elimination                   │
│                                                 │
└─────────────────────────────────────────────────┘
```

**Central compartment** = plasma + highly perfused organs (liver, kidneys)
**Peripheral compartment** = muscle, fat, connective tissue

## The Math

$$\frac{dC_1}{dt} = -(k_{10} + k_{12})C_1 + k_{21}C_2 + \frac{\text{input}(t)}{V_1}$$

$$\frac{dC_2}{dt} = k_{12}C_1 - k_{21}C_2$$

| Symbol | Meaning | Units |
|--------|---------|-------|
| $k_{10}$ | Elimination from central | 1/hr |
| $k_{12}$ | Transfer: central → peripheral | 1/hr |
| $k_{21}$ | Transfer: peripheral → central | 1/hr |
| $V_1$ | Volume of central compartment | L |
| $C_1, C_2$ | Drug concentrations | mg/L |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from pk_model import solve_2cmt, terminal_half_life, distribution_half_life, clearance
from dosing import iv_bolus, iv_infusion, oral_dose, multi_dose, regular_dosing
from analysis import auc_trapz, cmax_tmax, pk_summary
from visualization import single_dose_plot, multi_dose_plot, route_comparison_plot, sensitivity_plot

print('PharmaSim loaded successfully!')

---
## Phase 1 — Core Model: IV Bolus, Single Dose

We start with the simplest case: an **IV bolus** — drug injected directly into the bloodstream.

### Parameters (Generic Drug)
These parameters approximate a mid-range antibiotic. We'll switch to real vancomycin parameters in the validation notebook.

In [ ]:
# ── Model parameters ─────────────────────────────────────────────
k10 = 0.12   # 1/hr — elimination (think: kidney filtration rate)
k12 = 0.30   # 1/hr — distribution to tissue (fast redistribution)
k21 = 0.08   # 1/hr — return from tissue (slow — tissue holds drug)
V1  = 10.0   # L   — central volume

# Derived parameters (no free variables — fully determined by above)
CL    = clearance(V1, k10)
t12_b = terminal_half_life(k10, k12, k21)
t12_a = distribution_half_life(k10, k12, k21)

print(f'Clearance (CL):            {CL:.2f} L/hr')
print(f'Distribution half-life:    {t12_a:.2f} hr  (α phase — fast)')
print(f'Terminal half-life (t½β):  {t12_b:.2f} hr  (β phase — slow)')
print(f'Steady state after ~:      {5 * t12_b:.1f} hr')

In [ ]:
# ── Solve the ODE ─────────────────────────────────────────────────
dose_mg = 1000
t_end   = 48
t_eval  = np.linspace(0, t_end, 2000)

input_fn = iv_bolus(dose_mg)
t, C1, C2 = solve_2cmt((0, t_end), t_eval, input_fn, k10, k12, k21, V1)

# ── Quick plot ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, C1, label='C1 (plasma)', color='#2563eb', linewidth=2)
ax.plot(t, C2, label='C2 (tissue)', color='#16a34a', linewidth=2, linestyle='--')
ax.set(xlabel='Time (hr)', ylabel='Concentration (mg/L)',
       title=f'IV Bolus {dose_mg} mg — Two-Compartment Model')
ax.legend(); ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

# Print PK summary
pk_summary(t, C1, C2)

### What you're seeing

**C1 (blue):** Starts at its peak immediately (IV bolus → instant delivery to blood), then drops in **two phases**:
1. **α (distribution) phase** — rapid early drop as drug distributes into tissue
2. **β (elimination) phase** — slower late drop as drug is cleared by kidneys/liver

**C2 (green):** Starts at zero, rises as drug enters tissue, then falls as elimination drives drug back from tissue into plasma and out.

The tissue **peak comes later** than plasma — this is the hallmark of a two-compartment drug.

---
## Phase 2 — Semi-log Plot: Seeing the Biexponential

On a log-scale, a one-compartment drug would be a straight line.
A two-compartment drug shows **two distinct slopes** — the α and β phases.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Linear scale
ax1.plot(t, C1, color='#2563eb', linewidth=2)
ax1.set(title='Linear Scale', xlabel='Time (hr)', ylabel='C1 (mg/L)')
ax1.set_ylim(bottom=0)

# Semi-log scale — reveals the two phases
mask = C1 > 1e-4
ax2.semilogy(t[mask], C1[mask], color='#2563eb', linewidth=2)
ax2.set(title='Semi-log Scale (biexponential visible)',
        xlabel='Time (hr)', ylabel='log C1 (mg/L)')

# Annotate the two phases
ax2.annotate('α phase\n(distribution)\nsteep slope',
             xy=(2, C1[100]), xytext=(5, C1[50] * 3),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
ax2.annotate('β phase\n(elimination)\nshallower slope',
             xy=(30, C1[1200]), xytext=(20, C1[1200] * 0.2),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)

plt.tight_layout(); plt.show()

---
## Phase 3 — Publication Plot

This is the **README hero image** — a four-panel annotated figure.

In [ ]:
fig = single_dose_plot(k10=k10, k12=k12, k21=k21, V1=V1,
                       dose_mg=1000, route='iv_bolus',
                       t_end=48, save_path='single_dose.png')

---
## Phase 4 — Dosing Routes

Same dose, different route → very different concentration-time profiles.

In [ ]:
fig = route_comparison_plot(k10=k10, k12=k12, k21=k21, V1=V1,
                            dose_mg=1000, ka=1.2, F=0.85,
                            infusion_hr=1.0, t_end=36)

### Clinical translation

| Route | Cmax | AUC | Clinical use |
|-------|------|-----|--------------|
| IV bolus | Highest | = IV infusion | Emergency, anesthesia |
| IV infusion | Lower peak, same AUC | = IV bolus | Vancomycin, chemotherapy |
| Oral | Lowest (F<1) | < IV | Most outpatient drugs |

**AUC = total drug exposure.** For IV routes (100% bioavailable), AUC is identical regardless of infusion rate. The route only changes the *shape* of the curve, not total exposure.

---
## Phase 5 — Multiple Dosing & Steady State

In [ ]:
from analysis import accumulation_index

interval_hr = 12
R = accumulation_index(t12_b, interval_hr)

print(f'Terminal half-life:  {t12_b:.1f} hr')
print(f'Dose interval (τ):  {interval_hr} hr')
print(f'Accumulation index: R = {R:.2f}')
print(f'→ At steady state, peak is {R:.1f}× higher than after a single dose')
print(f'→ Steady state reached after ~{5 * t12_b:.0f} hr ({5 * t12_b / interval_hr:.1f} doses)')

In [ ]:
fig = multi_dose_plot(k10=k10, k12=k12, k21=k21, V1=V1,
                      dose_mg=1000, n_doses=10, interval_hr=12,
                      route='iv_bolus')

---
## Phase 6 — Parameter Sensitivity

Vary each rate constant and see how it reshapes the curve. This builds intuition for fitting real patient data.

In [ ]:
# How does changing elimination rate (k10) affect the profile?
fig = sensitivity_plot(param='k10', k10=k10, k12=k12, k21=k21, V1=V1,
                       dose_mg=1000, t_end=48)

In [ ]:
# How does tissue distribution speed (k12) affect the profile?
fig = sensitivity_plot(param='k12', k10=k10, k12=k12, k21=k21, V1=V1,
                       dose_mg=1000, t_end=48)

---
## Phase 7 — Interactive Widget (requires ipywidgets)

Run this cell to get live sliders. Move a slider → curve updates in real time.

In [1]:
try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider
    from IPython.display import display

    def interactive_pk(k10=0.12, k12=0.30, k21=0.08, V1=10.0, dose_mg=1000):
        t_eval = np.linspace(0, 48, 1000)
        input_fn = iv_bolus(dose_mg)
        t, C1, C2 = solve_2cmt((0, 48), t_eval, input_fn, k10, k12, k21, V1)

        cmax1, tmax1 = cmax_tmax(t, C1)
        auc1 = auc_trapz(t, C1)
        t12 = terminal_half_life(k10, k12, k21)

        fig, ax = plt.subplots(figsize=(11, 5))
        ax.plot(t, C1, color='#2563eb', linewidth=2, label='C1 (plasma)')
        ax.plot(t, C2, color='#16a34a', linewidth=2, linestyle='--', label='C2 (tissue)')
        ax.set(xlabel='Time (hr)', ylabel='Concentration (mg/L)',
               title=f'Cmax={cmax1:.2f}  Tmax={tmax1:.1f} hr  AUC={auc1:.0f}  t½β={t12:.2f} hr')
        ax.legend(); ax.set_ylim(bottom=0); ax.set_xlim(0, 48)
        plt.tight_layout(); plt.show()

    interact(interactive_pk,
             k10=FloatSlider(min=0.02, max=0.5,  step=0.01, value=0.12, description='k10 (elim)'),
             k12=FloatSlider(min=0.05, max=1.0,  step=0.05, value=0.30, description='k12 (→tissue)'),
             k21=FloatSlider(min=0.02, max=0.5,  step=0.01, value=0.08, description='k21 (→plasma)'),
             V1 =FloatSlider(min=2,    max=50,   step=1,    value=10,   description='V1 (L)'),
             dose_mg=FloatSlider(min=100, max=2000, step=100, value=1000, description='Dose (mg)'))

except ImportError:
    print('ipywidgets not installed. Run: pip install ipywidgets')
    print('Then restart the kernel and re-run this cell.')

ipywidgets not installed. Run: pip install ipywidgets
Then restart the kernel and re-run this cell.


---
## Summary — What You've Built

| Component | File | What it does |
|-----------|------|--------------|
| ODE system | `pk_model.py` | Two-compartment DEs, RK45 solver, macro constants |
| Dosing | `dosing.py` | IV bolus, IV infusion, oral, multi-dose superposition |
| Analysis | `analysis.py` | AUC, Cmax, Tmax, t½, Css, accumulation index |
| Visualization | `visualization.py` | 4-panel plots, route comparison, sensitivity spider |
| Validation | `validation/` | Vancomycin literature comparison |

**Next:** Open `validation/vancomycin_comparison.ipynb` to validate against published clinical data.